# state_conditioned_checkpoint_replay_v1

Purpose: replay the retained `state_conditioned_drawdown_rebound_reward_train_v1` checkpoint actors under the official backtest path before launching another training branch.

Governance:
- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `state_conditioned_overlay_gap_to_20_v1` and `soxlbacktest_reference_model_score_v1`.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`, plus 2026 YTD through the dashboard replay safe end.
- Changed layer: `model_selection` / `checkpoint_selection`.
- Acceptance: every walk-forward window must have an official-replay checkpoint with MaxDD `< 20%`, and the 2026 YTD checkpoint must match both the SOXLBacktest qlib reference return and the current-best return alignment threshold.
- Failure mode: early/low-risk checkpoints control drawdown but underparticipate; later/high-return checkpoints keep YTD upside but exceed the drawdown cap.

This is not a promotion notebook. It decides whether selection alone can recover a retained candidate, or whether the next branch must change reward/training behavior.


In [1]:

from __future__ import annotations

import contextlib
import io
import json
import multiprocessing as mp
from datetime import datetime, timezone
from pathlib import Path
import sys
import os

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

try:
    mp.set_start_method("fork", force=True)
except RuntimeError:
    pass

from src.soxl_quant.app.workers import WALK_FORWARD_WINDOWS
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.settings import QLIB_DATA_PATH, load_profile

EXPERIMENT = "state_conditioned_checkpoint_replay_v1"
PROFILE = "state_conditioned_drawdown_rebound_reward_train_v1"
CARD = Path("runtime/model_selection/card") / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
MODEL_CARD = Path("runtime/model/card") / PROFILE
REFERENCE_CARD = Path("runtime/model_selection/card/soxlbacktest_reference_model_score_v1/summary.json")
FRESHNESS = Path("dashboard_data/model_replay_freshness.json")

created_at = datetime.now(timezone.utc).isoformat()
reference_summary = json.loads(REFERENCE_CARD.read_text()) if REFERENCE_CARD.exists() else {}
thresholds = dict(reference_summary.get("thresholds", {}))
max_wf_dd_pct = float(thresholds.get("max_walkforward_drawdown_pct", 20.0))
required_soxlbacktest_return_pct = float(thresholds.get("required_soxlbacktest_return_pct", 132.077978))
required_best_return_pct = float(thresholds.get("required_best_return_pct", 213.143462))
ytd_start = str(thresholds.get("ytd_start", "2026-01-02"))
freshness = json.loads(FRESHNESS.read_text()) if FRESHNESS.exists() else {}
ytd_end = str(thresholds.get("ytd_end") or freshness.get("safe_backtest_end") or "2026-06-09")

init_qlib(QLIB_DATA_PATH)
profile = load_profile(PROFILE)
dataset = build_dataset(
    feature_packs=tuple(profile.feature.packs),
    profile_name=profile.name,
    test_seg=("2020-01-01", "2026-12-31"),
)
full_df = pd.concat(
    [segment_frame(dataset, "train"), segment_frame(dataset, "valid"), segment_frame(dataset, "test")]
).sort_index()
price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)

def actor_specs() -> list[dict[str, object]]:
    specs: list[dict[str, object]] = []
    for wf_name, train_seg, valid_seg, test_seg in WALK_FORWARD_WINDOWS:
        base = f"soxl_{PROFILE}_{wf_name}"
        fold_specs = [
            ("selected", MODEL_CARD / f"{base}_actor.pt"),
            ("iter0", MODEL_CARD / f"{base}_checkpoint_iter0_actor.pt"),
            ("iter4", MODEL_CARD / f"{base}_checkpoint_iter4_actor.pt"),
            ("iter8", MODEL_CARD / f"{base}_checkpoint_iter8_actor.pt"),
        ]
        for checkpoint, path in fold_specs:
            if path.exists():
                specs.append(
                    {
                        "evaluation_scope": "walk_forward",
                        "window": wf_name,
                        "candidate": f"{wf_name}:{checkpoint}",
                        "checkpoint": checkpoint,
                        "path": path,
                        "start": test_seg[0],
                        "end": test_seg[1],
                        "leakage_note": "fold-local actor evaluated only on its original isolated test window",
                    }
                )
    main_specs = [
        ("selected", MODEL_CARD / f"soxl_{PROFILE}_actor.pt"),
        ("iter0", MODEL_CARD / f"soxl_{PROFILE}_checkpoint_iter0_actor.pt"),
        ("iter4", MODEL_CARD / f"soxl_{PROFILE}_checkpoint_iter4_actor.pt"),
        ("iter8", MODEL_CARD / f"soxl_{PROFILE}_checkpoint_iter8_actor.pt"),
    ]
    for checkpoint, path in main_specs:
        if path.exists():
            specs.append(
                {
                    "evaluation_scope": "ytd_2026",
                    "window": "ytd_2026",
                    "candidate": f"ytd_2026:{checkpoint}",
                    "checkpoint": checkpoint,
                    "path": path,
                    "start": ytd_start,
                    "end": ytd_end,
                    "leakage_note": "main actor/checkpoint evaluated only for current YTD alignment",
                }
            )
    return specs

def load_actor(path: Path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    _, actor = build_policy(
        int(ckpt["obs_dim"]),
        profile,
        use_moe=bool(ckpt.get("use_moe", False)),
        hard_regime_routing=bool(ckpt.get("use_moe", False)),
    )
    actor.load_state_dict(ckpt["state_dict"])
    actor.eval()
    return actor

def compute_drawdown_interval(account: pd.Series) -> dict[str, object]:
    account = account.astype(float)
    peak = account.cummax()
    dd = (peak - account) / peak * 100.0
    if dd.empty:
        return {"peak_date": "", "trough_date": "", "interval_max_dd_pct": 0.0}
    trough = dd.idxmax()
    peak_date = account.loc[:trough].idxmax()
    def _date(value):
        return value.strftime("%Y-%m-%d") if hasattr(value, "strftime") else str(value)
    return {
        "peak_date": _date(peak_date),
        "trough_date": _date(trough),
        "interval_max_dd_pct": round(float(dd.loc[trough]), 6),
    }

def evaluate_spec(spec: dict[str, object]) -> dict[str, object]:
    start = str(spec["start"])
    end = str(spec["end"])
    feature_df = full_df.loc[start:end].copy()
    price = price_full.loc[start:end].copy()
    actor = load_actor(Path(spec["path"]))
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer), contextlib.redirect_stderr(buffer):
        bt = backtest_profile(
            actor=actor,
            feature_df=feature_df,
            price_series=price,
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            feature_columns=feature_columns,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=(start, end),
        )
    portfolio = bt["portfolio"]
    account = portfolio["account"].astype(float)
    drawdown_interval = compute_drawdown_interval(account)
    price_seg = price.loc[account.index.min():account.index.max()] if len(account) else price
    buyhold_return = float((price_seg.iloc[-1] / price_seg.iloc[0] - 1.0) * 100.0) if len(price_seg) > 1 else np.nan
    position_history = pd.Series(bt.get("position_history", []), dtype=float)
    avg_pos = float(position_history.mean()) if not position_history.empty else np.nan
    max_pos = float(position_history.max()) if not position_history.empty else np.nan
    row = {
        "experiment": EXPERIMENT,
        "profile": PROFILE,
        "evaluation_scope": spec["evaluation_scope"],
        "window": spec["window"],
        "candidate": spec["candidate"],
        "checkpoint": spec["checkpoint"],
        "actor_path": str(spec["path"]),
        "start": start,
        "end": end,
        "return_pct": round(float(bt["return"]), 6),
        "buyhold_return_pct": round(buyhold_return, 6),
        "relative_return_pct": round(float(bt["return"]) - buyhold_return, 6),
        "sharpe": round(float(bt["sharpe"]), 6),
        "max_drawdown_pct": round(float(bt["max_dd"]), 6),
        "trades": int(bt.get("trades", 0)),
        "avg_position": round(avg_pos, 6) if np.isfinite(avg_pos) else np.nan,
        "max_position": round(max_pos, 6) if np.isfinite(max_pos) else np.nan,
        "leakage_note": spec["leakage_note"],
        **drawdown_interval,
    }
    row["wf_drawdown_pass"] = bool(row["max_drawdown_pct"] < max_wf_dd_pct) if row["evaluation_scope"] == "walk_forward" else pd.NA
    row["ytd_soxlbacktest_return_pass"] = bool(row["return_pct"] >= required_soxlbacktest_return_pct) if row["evaluation_scope"] == "ytd_2026" else pd.NA
    row["ytd_current_best_alignment_pass"] = bool(row["return_pct"] >= required_best_return_pct) if row["evaluation_scope"] == "ytd_2026" else pd.NA
    return row

specs = actor_specs()
if not specs:
    raise RuntimeError(f"no actor checkpoints found under {MODEL_CARD}")
rows = []
for spec in specs:
    rows.append(evaluate_spec(spec))
metrics = pd.DataFrame(rows)
metrics.to_csv(CARD / "checkpoint_replay_metrics.csv", index=False)

wf_metrics = metrics[metrics["evaluation_scope"].eq("walk_forward")].copy()
ytd_metrics = metrics[metrics["evaluation_scope"].eq("ytd_2026")].copy()

fold_rows = []
for window, sub in wf_metrics.groupby("window", sort=True):
    ranked = sub.assign(
        dd_gap=lambda x: np.maximum(0.0, x["max_drawdown_pct"].astype(float) - max_wf_dd_pct),
        neg_return=lambda x: -x["return_pct"].astype(float),
    ).sort_values(["wf_drawdown_pass", "dd_gap", "neg_return", "candidate"], ascending=[False, True, True, True])
    best = ranked.iloc[0].to_dict()
    best["selected_for_window"] = True
    fold_rows.append(best)
fold_selection = pd.DataFrame(fold_rows)
fold_selection.to_csv(CARD / "fold_checkpoint_selection.csv", index=False)

ytd_selection = ytd_metrics.assign(
    both_reference_pass=lambda x: x["ytd_soxlbacktest_return_pass"].astype(bool) & x["ytd_current_best_alignment_pass"].astype(bool),
    neg_return=lambda x: -x["return_pct"].astype(float),
).sort_values(["both_reference_pass", "ytd_soxlbacktest_return_pass", "ytd_current_best_alignment_pass", "neg_return", "candidate"], ascending=[False, False, False, True, True])
ytd_selection.to_csv(CARD / "ytd_checkpoint_selection.csv", index=False)

best_ytd = ytd_selection.iloc[0].to_dict() if not ytd_selection.empty else {}
all_wf_drawdown_pass = bool(fold_selection["wf_drawdown_pass"].astype(bool).all()) if not fold_selection.empty else False
ytd_soxlbacktest_pass = bool(best_ytd.get("ytd_soxlbacktest_return_pass", False))
ytd_current_best_pass = bool(best_ytd.get("ytd_current_best_alignment_pass", False))
retention_gate_pass = bool(all_wf_drawdown_pass and ytd_soxlbacktest_pass and ytd_current_best_pass)

if retention_gate_pass:
    decision = "checkpoint_selector_candidate_ready_for_full_replay_review"
    next_action = "run full fixed-split plus walk-forward replay and registry promotion review before retaining actor"
elif all_wf_drawdown_pass and ytd_soxlbacktest_pass and not ytd_current_best_pass:
    decision = "checkpoint_selector_controls_wf_dd_but_misses_current_best_ytd_alignment"
    next_action = "change reward/training objective to preserve YTD upside while keeping the fold-local low-drawdown behavior"
elif all_wf_drawdown_pass and not ytd_soxlbacktest_pass:
    decision = "checkpoint_selector_controls_wf_dd_but_misses_soxlbacktest_reference_ytd"
    next_action = "do not retain checkpoint selector; retrain with stronger trend/upside preservation rather than lower caps"
else:
    decision = "checkpoint_selector_no_retention_candidate"
    next_action = "retrain or modify objective; selection-only replay did not satisfy walk-forward drawdown and YTD alignment gates"

summary = {
    "experiment": EXPERIMENT,
    "profile": PROFILE,
    "created_at": created_at,
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": retention_gate_pass,
    "retention_gate_pass": retention_gate_pass,
    "all_wf_drawdown_pass": all_wf_drawdown_pass,
    "ytd_soxlbacktest_return_pass": ytd_soxlbacktest_pass,
    "ytd_current_best_alignment_pass": ytd_current_best_pass,
    "selected_fold_checkpoints": fold_selection[["window", "candidate", "return_pct", "max_drawdown_pct", "wf_drawdown_pass"]].to_dict("records"),
    "selected_ytd_checkpoint": {k: best_ytd.get(k) for k in ["candidate", "return_pct", "max_drawdown_pct", "ytd_soxlbacktest_return_pass", "ytd_current_best_alignment_pass"]},
    "thresholds": {
        "max_walkforward_drawdown_pct": max_wf_dd_pct,
        "required_soxlbacktest_return_pct": required_soxlbacktest_return_pct,
        "required_best_return_pct": required_best_return_pct,
        "ytd_start": ytd_start,
        "ytd_end": ytd_end,
    },
    "next_action": next_action,
    "sidecars": {
        "checkpoint_replay_metrics": str(CARD / "checkpoint_replay_metrics.csv"),
        "fold_checkpoint_selection": str(CARD / "fold_checkpoint_selection.csv"),
        "ytd_checkpoint_selection": str(CARD / "ytd_checkpoint_selection.csv"),
        "selection": str(CARD / "selection.json"),
        "summary": str(CARD / "summary.json"),
        "conclusion": str(CARD / "conclusion.md"),
    },
}
selection = {
    **summary,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": ["state_conditioned_overlay_gap_to_20_v1", "soxlbacktest_reference_model_score_v1"],
    "role": "diagnose",
    "changed_layer": ["model_selection", "checkpoint_selection"],
    "target_window": ["wf_2020", "wf_2022", "wf_2023_2024", "ytd_2026"],
    "accepted_tradeoff": "No actor retention unless every WF replay is below 20% MaxDD and YTD return clears both reference thresholds.",
    "hypothesis": "Retained fold-local checkpoints may contain a lower-drawdown candidate that the scalar validation score did not select.",
    "failure_mode": "Low-risk retained checkpoints underparticipate; high-return checkpoints preserve YTD but fail WF drawdown.",
    "duplicate_check": "This does not add another cap or rerun PPO; it replays existing retained checkpoint actors under the official backtest path.",
    "external_reference": [
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://arxiv.org/abs/2011.09607",
    ],
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, allow_nan=False), encoding="utf-8")

wf_lines = []
for row in summary["selected_fold_checkpoints"]:
    wf_lines.append(
        f"- {row['window']}: `{row['candidate']}` return `{row['return_pct']:.2f}%`, MaxDD `{row['max_drawdown_pct']:.2f}%`, pass={row['wf_drawdown_pass']}"
    )
ytd_line = summary["selected_ytd_checkpoint"]
conclusion = "\n".join(
    [
        f"# {EXPERIMENT}",
        "",
        f"- Decision: `{decision}`.",
        f"- Actor retention allowed: `{retention_gate_pass}`.",
        f"- WF drawdown gate: `{all_wf_drawdown_pass}` using MaxDD `< {max_wf_dd_pct:.2f}%`.",
        f"- YTD SOXLBacktest reference gate: `{ytd_soxlbacktest_pass}` using return `>= {required_soxlbacktest_return_pct:.2f}%`.",
        f"- YTD current-best alignment gate: `{ytd_current_best_pass}` using return `>= {required_best_return_pct:.2f}%`.",
        "",
        "## Selected Fold Checkpoints",
        *wf_lines,
        "",
        "## Selected YTD Checkpoint",
        f"- `{ytd_line.get('candidate', '')}` return `{float(ytd_line.get('return_pct', float('nan'))):.2f}%`, MaxDD `{float(ytd_line.get('max_drawdown_pct', float('nan'))):.2f}%`, SOXLBacktest pass={ytd_line.get('ytd_soxlbacktest_return_pass')}, current-best pass={ytd_line.get('ytd_current_best_alignment_pass')}.",
        "",
        "## Next Action",
        f"- {next_action}",
        "",
        "## Sidecars",
        "- `checkpoint_replay_metrics.csv`",
        "- `fold_checkpoint_selection.csv`",
        "- `ytd_checkpoint_selection.csv`",
        "- `selection.json`",
        "- `summary.json`",
    ]
).rstrip() + "\n"
(CARD / "conclusion.md").write_text(conclusion, encoding="utf-8")

display(Markdown(conclusion))
metrics


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[69943:MainThread](2026-06-13 19:22:10,104) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[69943:MainThread](2026-06-13 19:22:10,544) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[69943:MainThread](2026-06-13 19:22:10,545) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[69943:MainThread](2026-06-13 19:22:12,076) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

[69943:MainThread](2026-06-13 19:22:18,521) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


[69943:MainThread](2026-06-13 19:22:18,521) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/108 [00:00<?, ?it/s]

# state_conditioned_checkpoint_replay_v1

- Decision: `checkpoint_selector_no_retention_candidate`.
- Actor retention allowed: `False`.
- WF drawdown gate: `False` using MaxDD `< 20.00%`.
- YTD SOXLBacktest reference gate: `False` using return `>= 132.08%`.
- YTD current-best alignment gate: `False` using return `>= 213.14%`.

## Selected Fold Checkpoints
- wf_2020: `wf_2020:iter0` return `41.16%`, MaxDD `19.25%`, pass=True
- wf_2022: `wf_2022:iter0` return `-20.24%`, MaxDD `21.65%`, pass=False
- wf_2023_2024: `wf_2023_2024:iter8` return `1.51%`, MaxDD `6.97%`, pass=True

## Selected YTD Checkpoint
- `ytd_2026:iter8` return `77.97%`, MaxDD `15.67%`, SOXLBacktest pass=False, current-best pass=False.

## Next Action
- retrain or modify objective; selection-only replay did not satisfy walk-forward drawdown and YTD alignment gates

## Sidecars
- `checkpoint_replay_metrics.csv`
- `fold_checkpoint_selection.csv`
- `ytd_checkpoint_selection.csv`
- `selection.json`
- `summary.json`


,experiment,profile,evaluation_scope,window,candidate,checkpoint,actor_path,start,end,return_pct,...,trades,avg_position,max_position,leakage_note,peak_date,trough_date,interval_max_dd_pct,wf_drawdown_pass,ytd_soxlbacktest_return_pass,ytd_current_best_alignment_pass
0,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2020,wf_2020:selected,selected,runtime/model/card/state_conditioned_drawdown_...,2020-01-01,2020-12-31,81.255715,...,37,0.428601,0.880458,fold-local actor evaluated only on its origina...,2020-09-02,2020-09-08,24.414034,False,<NA>,<NA>
1,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2020,wf_2020:iter0,iter0,runtime/model/card/state_conditioned_drawdown_...,2020-01-01,2020-12-31,41.161806,...,28,0.260518,0.645381,fold-local actor evaluated only on its origina...,2020-09-02,2020-09-08,19.252065,True,<NA>,<NA>
2,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2020,wf_2020:iter4,iter4,runtime/model/card/state_conditioned_drawdown_...,2020-01-01,2020-12-31,56.767330,...,32,0.345377,0.760120,fold-local actor evaluated only on its origina...,2020-09-02,2020-09-08,21.894289,False,<NA>,<NA>
3,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2020,wf_2020:iter8,iter8,runtime/model/card/state_conditioned_drawdown_...,2020-01-01,2020-12-31,81.255715,...,37,0.428601,0.880458,fold-local actor evaluated only on its origina...,2020-09-02,2020-09-08,24.414034,False,<NA>,<NA>
4,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2022,wf_2022:selected,selected,runtime/model/card/state_conditioned_drawdown_...,2022-01-01,2022-12-31,-33.669921,...,33,0.268742,0.626487,fold-local actor evaluated only on its origina...,2022-03-30,2022-11-03,36.178874,False,<NA>,<NA>
5,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2022,wf_2022:iter0,iter0,runtime/model/card/state_conditioned_drawdown_...,2022-01-01,2022-12-31,-20.241869,...,22,0.133036,0.441605,fold-local actor evaluated only on its origina...,2022-03-30,2022-12-28,21.649887,False,<NA>,<NA>
6,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2022,wf_2022:iter4,iter4,runtime/model/card/state_conditioned_drawdown_...,2022-01-01,2022-12-31,-33.669921,...,33,0.268742,0.626487,fold-local actor evaluated only on its origina...,2022-03-30,2022-11-03,36.178874,False,<NA>,<NA>
7,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2022,wf_2022:iter8,iter8,runtime/model/card/state_conditioned_drawdown_...,2022-01-01,2022-12-31,-33.669921,...,33,0.268742,0.626487,fold-local actor evaluated only on its origina...,2022-03-30,2022-11-03,36.178874,False,<NA>,<NA>
8,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2023_2024,wf_2023_2024:selected,selected,runtime/model/card/state_conditioned_drawdown_...,2024-01-01,2024-12-31,-4.162874,...,30,0.151507,0.412500,fold-local actor evaluated only on its origina...,2024-07-10,2024-12-19,16.033688,True,<NA>,<NA>
9,state_conditioned_checkpoint_replay_v1,state_conditioned_drawdown_rebound_reward_trai...,walk_forward,wf_2023_2024,wf_2023_2024:iter0,iter0,runtime/model/card/state_conditioned_drawdown_...,2024-01-01,2024-12-31,-4.162874,...,30,0.151507,0.412500,fold-local actor evaluated only on its origina...,2024-07-10,2024-12-19,16.033688,True,<NA>,<NA>
